In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

def build_gold_cliente_mes():
    print("Construindo a gold_cliente_mes com suporte a dados atrasados (Late-Arriving Data)...")
    
    query = """
    CREATE OR REPLACE TABLE workspace.default.gold_cliente_mes AS
    
    WITH agregacao_mensal AS (
        SELECT 
            c.id_cliente,
            c.segmento,
            DATE_TRUNC('month', f.data_transacao) AS mes_referencia,
            SUM(f.valor_liquido) AS gasto_total_mes,
            COUNT(f.id_transacao) AS qtd_transacoes_mes
        FROM workspace.default.gold_fato_transacao f
        -- Joins para chegar do cartão até o cliente
        JOIN workspace.default.silver_cartoes cart 
            ON f.id_cartao = cart.id_cartao
        JOIN workspace.default.silver_contas cont 
            ON cart.id_conta = cont.id_conta
        JOIN workspace.default.silver_clientes c 
            ON cont.id_cliente = c.id_cliente
        WHERE f.is_estornada = FALSE
        GROUP BY 1, 2, 3
    )
    
    -- CTE Final usando LAG para comparar com o mês anterior
    SELECT 
        id_cliente,
        segmento,
        CAST(mes_referencia AS DATE) AS mes_referencia,
        gasto_total_mes,
        qtd_transacoes_mes,
        
        -- Busca o gasto do mês imediatamente anterior
        LAG(gasto_total_mes) OVER (
            PARTITION BY id_cliente 
            ORDER BY mes_referencia
        ) AS gasto_mes_anterior,
        
        -- Calcula a variação percentual (cuidando com a divisão por zero)
        ROUND(
            ((gasto_total_mes - LAG(gasto_total_mes) OVER (PARTITION BY id_cliente ORDER BY mes_referencia)) 
            / NULLIF(LAG(gasto_total_mes) OVER (PARTITION BY id_cliente ORDER BY mes_referencia), 0)) * 100, 
        2) AS variacao_gasto_pct

    FROM agregacao_mensal;
    """
    
    spark.sql(query)
    print("Sucesso! Tabela workspace.default.gold_cliente_mes criada e tratada para dados históricos.")

# Executa e exibe uma amostra
build_gold_cliente_mes()
display(spark.sql("SELECT * FROM workspace.default.gold_cliente_mes ORDER BY id_cliente, mes_referencia LIMIT 10"))